In [1]:

from pathlib import Path

print("Current folder:", Path.cwd())
print("\nZIP files found:")

for f in Path.cwd().glob("*.zip"):
    print(f.name, "-", round(f.stat().st_size / 1024, 1), "KB")


Current folder: /workspace

ZIP files found:


In [2]:

from pathlib import Path

print("Searching workspace for the uploaded ZIP...")

matches = list(Path("/workspace").rglob("twinmind_sensor_data.zip"))

if matches:
    for path in matches:
        print("FOUND:", path)
        print("Size:", round(path.stat().st_size / 1024, 1), "KB")
else:
    print("ZIP not found under /workspace.")
    print("Top-level workspace items:")
    for item in Path("/workspace").iterdir():
        print(item.name, "(folder)" if item.is_dir() else "(file)")


Searching workspace for the uploaded ZIP...
FOUND: /workspace/twinmind_amd_results/twinmind_sensor_data.zip
Size: 599.5 KB


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("ROCm/HIP:", torch.version.hip)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable — stop here and share this output.")


PyTorch: 2.13.0+rocm10.0.0
ROCm/HIP: 7.15.26333
GPU available: True
GPU name: AMD Radeon Pro W7900D


In [4]:

from pathlib import Path
import zipfile
import pandas as pd

zip_path = Path("/workspace/twinmind_amd_results/twinmind_sensor_data.zip")
extract_dir = Path("/workspace/twinmind_amd_results/sensor_csvs")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

csv_files = sorted(extract_dir.rglob("*.csv"))

print("CSV files found:", len(csv_files))
for f in csv_files:
    print(f.name, "-", round(f.stat().st_size / 1024, 1), "KB")

assert csv_files, "No CSV files found inside the ZIP."


CSV files found: 9
CombinedFaults_02222025.csv - 2298.6 KB
Combined_Baseline1.csv - 297.3 KB
Combined_Baseline2.csv - 299.6 KB
Combined_BentShaft.csv - 294.2 KB
Combined_EccentricRotor.csv - 365.2 KB
Combined_FaultedBearing.csv - 255.5 KB
Combined_FaultedCoupling.csv - 224.8 KB
Combined_Imbalance.csv - 449.1 KB
Combined_QualityTest.csv - 300.8 KB


In [5]:

import torch

print("PyTorch:", torch.__version__)
print("ROCm/HIP:", torch.version.hip)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    raise RuntimeError(
        "GPU unavailable. Stop and share this output before proceeding."
    )


PyTorch: 2.13.0+rocm10.0.0
ROCm/HIP: 7.15.26333
GPU available: True
GPU name: AMD Radeon Pro W7900D
GPU count: 1


In [6]:

import numpy as np
import pandas as pd
import torch
import time
from pathlib import Path

FEATURES = [
    "ch1_bias", "ch1_derivedPk", "ch1_direct",
    "ch1_directRMS", "ch1_velocityPk", "ch1_velocityRMS",
    "ch2_bias", "ch2_derivedPk", "ch2_direct",
    "ch2_directRMS", "ch2_velocityPk", "ch2_velocityRMS"
]

data_dir = Path("/workspace/twinmind_amd_results/sensor_csvs")
frames = []

for file in sorted(data_dir.rglob("*.csv")):
    df = pd.read_csv(file)
    df.columns = df.columns.astype(str).str.strip()

    if not any(c in df.columns for c in FEATURES):
        print("Skipping file without expected sensor columns:", file.name)
        continue

    sensor = df.reindex(columns=FEATURES)
    for col in FEATURES:
        sensor[col] = pd.to_numeric(sensor[col], errors="coerce")

    sensor["source_file"] = file.name
    sensor["source_row"] = np.arange(len(sensor))
    frames.append(sensor)

assert frames, "No files contain the expected TwinMind sensor features."

data = pd.concat(frames, ignore_index=True)
X_np = data[FEATURES].to_numpy(dtype=np.float32)

device = torch.device("cuda")
torch.cuda.synchronize()
start = time.perf_counter()

# Move the real sensor measurements to the AMD GPU.
X = torch.as_tensor(X_np, dtype=torch.float32, device=device)
finite = torch.isfinite(X)
X_nan = torch.where(finite, X, torch.nan)

# Impute missing/non-finite values with per-feature means.
means = torch.nanmean(X_nan, dim=0)
means = torch.nan_to_num(means, nan=0.0, posinf=0.0, neginf=0.0)
X_clean = torch.where(finite, X, means.unsqueeze(0))

# GPU-based standardization.
center = X_clean.mean(dim=0)
scale = X_clean.std(dim=0, unbiased=False)
scale = torch.where(scale > 1e-8, scale, torch.ones_like(scale))
Z = (X_clean - center) / scale

# Screening heuristic; this is not a confirmed fault diagnosis.
scores = Z.abs().mean(dim=1)
threshold = torch.quantile(scores, 0.95)
flagged = scores >= threshold

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print("GPU:", torch.cuda.get_device_name(0))
print("Sensor tensor device:", X.device)
print("Rows processed:", X.shape[0])
print("Features processed:", X.shape[1])
print("Missing/non-finite values:", int((~finite).sum().item()))
print("95th-percentile screening threshold:", round(threshold.item(), 4))
print("Samples flagged for review:", int(flagged.sum().item()))
print("GPU workload time (seconds):", round(elapsed, 3))


GPU: AMD Radeon Pro W7900D
Sensor tensor device: cuda:0
Rows processed: 45239
Features processed: 12
Missing/non-finite values: 481
95th-percentile screening threshold: 1.7293
Samples flagged for review: 2262
GPU workload time (seconds): 0.949


In [7]:

output_dir = Path("/workspace/twinmind_amd_results")
output_dir.mkdir(parents=True, exist_ok=True)

results = data[["source_file", "source_row"]].copy()
results["screening_score"] = scores.detach().cpu().numpy()
results["flagged_for_review"] = flagged.detach().cpu().numpy()

results.sort_values(
    "screening_score", ascending=False
).to_csv(
    output_dir / "twinmind_gpu_sensor_screening.csv",
    index=False
)

feature_summary = pd.DataFrame({
    "feature": FEATURES,
    "mean_after_imputation": center.detach().cpu().numpy(),
    "standard_deviation": scale.detach().cpu().numpy()
})
feature_summary.to_csv(
    output_dir / "twinmind_gpu_feature_summary.csv",
    index=False
)

print("Saved screening results and feature summary to:", output_dir)
print("\nTop 10 samples for review:")
print(results.nlargest(10, "screening_score").to_string(index=False))


Saved screening results and feature summary to: /workspace/twinmind_amd_results

Top 10 samples for review:
                source_file  source_row  screening_score  flagged_for_review
CombinedFaults_02222025.csv       19356         2.146849                True
     Combined_BentShaft.csv        1117         2.146849                True
CombinedFaults_02222025.csv       19388         2.139896                True
     Combined_BentShaft.csv        1149         2.139896                True
CombinedFaults_02222025.csv       20552         2.123130                True
     Combined_BentShaft.csv        2313         2.123130                True
CombinedFaults_02222025.csv       20570         2.093491                True
     Combined_BentShaft.csv        2331         2.093491                True
CombinedFaults_02222025.csv       20572         2.072847                True
     Combined_BentShaft.csv        2333         2.072847                True


In [8]:
from pathlib import Path

output_dir = Path("/workspace/twinmind_amd_results")

for name in [
    "twinmind_sensor_data.zip",
    "twinmind_gpu_sensor_screening.csv",
    "twinmind_gpu_feature_summary.csv",
]:
    matches = list(output_dir.rglob(name))
    print(("FOUND " if matches else "MISSING "), name)
    for path in matches:
        print("   ", path, f"({path.stat().st_size:,} bytes)")


FOUND  twinmind_sensor_data.zip
    /workspace/twinmind_amd_results/twinmind_sensor_data.zip (613,853 bytes)
FOUND  twinmind_gpu_sensor_screening.csv
    /workspace/twinmind_amd_results/twinmind_gpu_sensor_screening.csv (2,155,717 bytes)
FOUND  twinmind_gpu_feature_summary.csv
    /workspace/twinmind_amd_results/twinmind_gpu_feature_summary.csv (474 bytes)
